In [1]:
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/merged_dataset
!rm -rf /content/test_dataset
!unzip -q "/content/drive/MyDrive/MRP/merged_dataset.zip" -d "/content/merged_dataset"
!unzip -q "/content/drive/MyDrive/MRP/test_dataset.zip" -d "/content/test_dataset"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=32,
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=32,
    validation_split=0.2,
    subset="validation",
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/test_dataset/test_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=32
)

Found 351500 files belonging to 2 classes.
Using 281200 files for training.
Found 351500 files belonging to 2 classes.
Using 70300 files for validation.
Found 18124 files belonging to 2 classes.


In [4]:
def build_baseline_cnn(input_shape=(256, 256, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 4
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Classification head
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu', name="feature_vector"),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    return model

model = build_baseline_cnn()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [5]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
8788/8788 ━━━━━━━━━━━━━━━━━━━━ 600s 67ms/step - accuracy: 0.7440 - loss: 0.5219 - val_accuracy: 0.7851 - val_loss: 0.4636
Epoch 2/20
8788/8788 ━━━━━━━━━━━━━━━━━━━━ 572s 65ms/step - accuracy: 0.8424 - loss: 0.3748 - val_accuracy: 0.8700 - val_loss: 0.3190
Epoch 3/20
8788/8788 ━━━━━━━━━━━━━━━━━━━━ 571s 65ms/step - accuracy: 0.8870 - loss: 0.2795 - val_accuracy: 0.8796 - val_loss: 0.2951
Epoch 4/20
8788/8788 ━━━━━━━━━━━━━━━━━━━━ 571s 65ms/step - accuracy: 0.9048 - loss: 0.2344 - val_accuracy: 0.8509 - val_loss: 0.4444
Epoch 5/20
8788/8788 ━━━━━━━━━━━━━━━━━━━━ 571s 65ms/step - accuracy: 0.9176 - loss: 0.2025 - val_accuracy: 0.7906 - val_loss: 0.5481


In [7]:
loss, acc = model.evaluate(val_ds)
print("Validation accuracy:", acc)

2197/2197 ━━━━━━━━━━━━━━━━━━━━ 27s 12ms/step - accuracy: 0.8796 - loss: 0.2951
Validation accuracy: 0.8795590400695801


In [8]:
loss, acc = model.evaluate(test_ds)
print("Validation accuracy:", acc)

567/567 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6423 - loss: 0.8920
Validation accuracy: 0.6422975063323975


In [9]:
model.save("/content/drive/MyDrive/MRP/models/raw_cnn_baseline.h5")

In [10]:
# Note: After run:
# model = load_model("/content/drive/MyDrive/MRP/models/raw_cnn_baseline.h5")
# feature_extractor = tf.keras.Model(
#     inputs=model.input,
#     outputs=model.get_layer("feature_vector").output
# )